# Explore the xBD folder structure
 -------------------------------------------------------------
 Purpose : Print a compact tree of /kaggle/input so we can see
           where tier1 / tier3 / test / hold and their
           images / labels / masks folders actually live.
 
 Why     : The Kaggle dataset uses a nested, non-standard
           layout, so paths must be confirmed before use.
 
 Output  : For every folder -> number of files, file extension
           counts, and a few sample filenames. No GPU needed.

In [1]:
import os
from collections import Counter

INPUT_ROOT = "/kaggle/input"   # all Kaggle datasets are mounted here
MAX_DEPTH  = 6                 # how deep to walk into nested folders
N_SAMPLES  = 3                 # sample filenames to show per folder

def show_tree(root, max_depth=MAX_DEPTH, n_samples=N_SAMPLES):
    root_depth = root.rstrip(os.sep).count(os.sep)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.rstrip(os.sep).count(os.sep) - root_depth
        if depth > max_depth:
            dirnames[:] = []          # stop walking deeper
            continue
        dirnames.sort()               # print folders in a stable order
        indent = "    " * depth
        name = os.path.basename(dirpath) or dirpath

        if filenames:
            # Count files by extension, e.g. .png: 5598, .json: 5598
            exts = Counter(os.path.splitext(f)[1].lower() for f in filenames)
            ext_str = ", ".join(f"{e or 'no-ext'}: {c}" for e, c in exts.most_common())
            print(f"{indent}{name}/  [{len(filenames)} files | {ext_str}]")
            for f in sorted(filenames)[:n_samples]:
                print(f"{indent}    - {f}")
        else:
            print(f"{indent}{name}/")

show_tree(INPUT_ROOT)

input/
    datasets/
        qianlanzz/
            xbd-dataset/
                xbd/
                    hold/
                        images/  [1866 files | .png: 1866]
                            - guatemala-volcano_00000004_post_disaster.png
                            - guatemala-volcano_00000004_pre_disaster.png
                            - guatemala-volcano_00000012_post_disaster.png
                        labels/  [1866 files | .json: 1866]
                            - guatemala-volcano_00000004_post_disaster.json
                            - guatemala-volcano_00000004_pre_disaster.json
                            - guatemala-volcano_00000012_post_disaster.json
                        masks/  [2799 files | .png: 2799]
                            - guatemala-volcano_00000004_post_disaster.png
                            - guatemala-volcano_00000004_post_disaster_rgb.png
                            - guatemala-volcano_00000004_pre_disaster.png
                    test/
      

# Phase 1 - Step 1 (Cell 2): Locate split folders automatically
 -------------------------------------------------------------
 Purpose : Find the full path of each split (tier1, tier3,
           test, hold) wherever it sits in the nested layout,
           and list its subfolders (images / labels / masks)
           with file counts.

 Output  : SPLIT_DIRS = {split_name: full_path}. All later
           steps reuse this dict instead of hardcoded paths.

In [2]:
from collections import defaultdict

SPLIT_NAMES = ["tier1", "tier3", "test", "hold"]

# Collect every folder whose name matches a split name
found = defaultdict(list)
for dirpath, dirnames, _ in os.walk(INPUT_ROOT):
    for d in dirnames:
        if d.lower() in SPLIT_NAMES:
            found[d.lower()].append(os.path.join(dirpath, d))

SPLIT_DIRS = {}
for split in SPLIT_NAMES:
    paths = found.get(split, [])

    if not paths:
        print(f"[MISSING] '{split}' folder was not found")
        continue

    if len(paths) > 1:
        # Same split name in several places -> show all, use the first
        print(f"[WARNING] '{split}' found in {len(paths)} places, using the first:")
        for p in paths:
            print("     ", p)

    SPLIT_DIRS[split] = paths[0]
    print(f"\n{split}: {paths[0]}")

    # Show subfolders inside the split and how many files each has
    for sub in sorted(os.listdir(paths[0])):
        sub_path = os.path.join(paths[0], sub)
        if os.path.isdir(sub_path):
            print(f"    {sub}/ -> {len(os.listdir(sub_path))} files")

print("\nSPLIT_DIRS =", SPLIT_DIRS)


tier1: /kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1
    images/ -> 5598 files
    labels/ -> 5598 files
    masks/ -> 8397 files

tier3: /kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier3
    images/ -> 12738 files
    labels/ -> 12738 files
    masks/ -> 19107 files

test: /kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/test
    images/ -> 1866 files
    labels/ -> 1866 files
    masks/ -> 2799 files

hold: /kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/hold
    images/ -> 1866 files
    labels/ -> 1866 files
    masks/ -> 2799 files

SPLIT_DIRS = {'tier1': '/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1', 'tier3': '/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier3', 'test': '/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/test', 'hold': '/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/hold'}
